# 第5回　相関と因果
## ―― 「相関がある」と「原因である」は、まったく違う

統計学Ⅰ（B）　／　北星学園大学

今日は**▶を上から押す**。ところどころ「やってみよう」で列名を変えて試せる。注目は ――

> 2つのものが一緒に動いても、**一方が他方の原因とは限らない。**

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## フック：この相関を信じる？

- 「**アイスの売上**が増えると、**水難事故**も増える」――強い相関がある。ではアイスを禁止すれば事故は減る？
- 「**チョコの消費量**が多い国ほど**ノーベル賞**受賞者が多い」――チョコを食べれば頭が良くなる？

どちらも「相関」は本物。でも結論はばかげている。**なぜか**を、今日のデータで解き明かす。

そして今日は、**ばかげているとすぐには分からない例**を扱う。そちらが本番である。

---
## 1. 散布図と相関係数 r

2つの量の関係は、**散布図**（点を打つ）で見るのが基本。その関係の強さを1つの数にしたのが **相関係数 r**（−1〜+1）。

- r が +1 に近い：片方が増えると、もう片方も増える（右上がり）
- r が −1 に近い：片方が増えると、もう片方は減る（右下がり）
- r が 0 に近い：直線的な関係がない

> **今日は体重・行動圏など、何桁にもまたがる量を扱う。**
> 第3回でやったとおり、こういう量は**対数の目盛りで見る**（そうしないと1種の大型類人猿が図を支配する）。

まず「体重 と 行動圏（ふだん動きまわる範囲）」を見てみよう。

In [ ]:
e = df.dropna(subset=["体重g", "行動圏km2"])
x, y = np.log10(e["体重g"]), np.log10(e["行動圏km2"])
r = x.corr(y)

print(f"体重 と 行動圏 の相関係数 r = {r:.3f}   (n={len(e)}種)")

plt.figure(figsize=(6.5,4.5))
plt.scatter(x, y, s=16, alpha=0.55, color="#00897b")
plt.xlabel("体重（対数目盛り）"); plt.ylabel("行動圏（対数目盛り）")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.yticks([-2,-1,0,1], ["0.01km²","0.1km²","1km²","10km²"])
plt.title(f"体が大きい種ほど広く動く（r={r:.2f}）")
plt.show()

r ≈ 0.68。右上がりで、体の大きい種ほど広い範囲を使っている。**大きい動物はたくさん食べる必要があるから広く動く** ―― これは**納得のいく**関係だ。

では次の相関はどうだろう？

---
## 2. 「乳離れが遅い種ほど、広く動きまわる」？

`離乳日齢`（子が乳を離れるまでの日数）と `行動圏km2` の相関を見る。

In [ ]:
e2 = df.dropna(subset=["離乳日齢", "行動圏km2"])
x2, y2 = np.log10(e2["離乳日齢"]), np.log10(e2["行動圏km2"])
r2 = x2.corr(y2)

print(f"離乳日齢 と 行動圏 の相関係数 r = {r2:.3f}   (n={len(e2)}種)")

plt.figure(figsize=(6.5,4.5))
plt.scatter(x2, y2, s=16, alpha=0.55, color="#e8503a")
plt.xlabel("離乳日齢（対数目盛り）"); plt.ylabel("行動圏（対数目盛り）")
plt.xticks([1.5,2,2.5,3,3.5], ["30日","100日","300日","1000日","3000日"])
plt.yticks([-2,-1,0,1], ["0.01km²","0.1km²","1km²","10km²"])
plt.title(f"乳離れが遅い種ほど広く動く？（r={r2:.2f}）")
plt.show()

**r ≈ 0.66。かなり強い。** さきほどの体重と行動圏（0.68）とほとんど変わらない。

「**子育て期間が長い種ほど、広い行動圏を必要とする**」――もっともらしい。生態学の論文の見出しになりそうだし、その気になれば理屈も後付けできる。「長く子を連れて歩くから広い範囲を使うのだろう」とか。

でも、本当に**離乳の遅さが原因**なのだろうか？　―― ここで立ち止まるのが統計の思考だ。

---
## 3. 交絡を疑え ―― 第三の変数

「乳離れが遅い種」とは、どんな種だろうか。**大きい種**である。ゾウもクジラもヒトも、大きい動物ほど子育てに時間がかかる。

そして第1節で見たとおり、**大きい種は行動圏も広い**。

まず、離乳日齢と体重がどれくらい結びついているかを確かめよう。

In [ ]:
e3 = df.dropna(subset=["離乳日齢", "体重g"])
r3 = np.log10(e3["離乳日齢"]).corr(np.log10(e3["体重g"]))
print(f"離乳日齢 と 体重 の相関 r = {r3:.3f}   (n={len(e3)}種)")
print("→ 乳離れが遅い種は、そもそも体が大きい")
print()

# 実際の値で見る
sub = df.dropna(subset=["離乳日齢", "行動圏km2", "体重g"])
print("【大きい種】")
print(sub.nlargest(3, "体重g")[["学名", "体重g", "離乳日齢", "行動圏km2"]].to_string(index=False))
print()
print("【小さい種】")
print(sub.nsmallest(3, "体重g")[["学名", "体重g", "離乳日齢", "行動圏km2"]].to_string(index=False))

r ≈ 0.85。**離乳日齢と体重は、ほとんど同じことを測っている**と言ってよいほど強い。

チンパンジー（45kg）は離乳まで1261日＝3年半かかり、行動圏は10.85km²。ネズミキツネザル（48g）は離乳40日、行動圏0.01km²。**どちらの列も、体の大きさが動かしている。**

```
        （体の大きさ）
         ／        ＼
   離乳が遅い      行動圏が広い
```

つまり離乳日齢は「大きい種がついでに持っている性質」かもしれない。これを **交絡（こうらく）** という。

確かめ方：**体重の影響を取り除いて**、離乳日齢と行動圏の関係が残るか見る（偏相関）。

In [ ]:
# 体重の効果を引き算してから、離乳日齢と行動圏の相関を測る
s = df.dropna(subset=["離乳日齢", "行動圏km2", "体重g"]).copy()
for col in ["離乳日齢", "行動圏km2", "体重g"]:
    s[col] = np.log10(s[col])

X = np.column_stack([np.ones(len(s)), s["体重g"]])
def 残差(v):
    b = np.linalg.lstsq(X, s[v], rcond=None)[0]
    return s[v] - X @ b

print(f"（同じ {len(s)} 種で比較）")
print(f"離乳日齢と行動圏：単純な相関   r = {s['離乳日齢'].corr(s['行動圏km2']):.3f}")
print(f"離乳日齢と行動圏：体重を統制後 r = {np.corrcoef(残差('離乳日齢'), 残差('行動圏km2'))[0,1]:.3f}")
print()
print("→ ほぼ 0。見かけの関係は、体の大きさによる交絡だった。")

**離乳の効果は消えた。** r = 0.66 が 0.05 になった。

アイスと水難事故（犯人は「気温」）、チョコとノーベル賞（犯人は「経済的豊かさ」）と同じ構造である。

> **相関を見たら、因果を結論する前に『隠れた第三の変数（交絡）』を疑え。**

### この分野では、これが最大の落とし穴である

生き物のデータでは、**体の大きさがほとんど全部に効く**。寿命も、妊娠期間も、行動圏も、集団サイズも、脳の重さも、大きい種ほど大きい。

だから種どうしを比べると、**何と何を組み合わせても相関が出る**。「Aが大きい種ほどBも大きい」という発見は、たいてい「どちらも体が大きいだけ」である。

先ほど見つけた組み合わせを、いくつも探せる。試してみよう。

In [ ]:
# いろいろな組み合わせで、体重を統制する前後を比べる
import itertools
cols = ["頭胴長mm", "新生児体重g", "離乳日齢", "妊娠期間日", "最長寿命月", "行動圏km2", "出産間隔日"]

print("組み合わせ                    単純r   体重統制後   n")
print("-" * 52)
for a, b in itertools.combinations(cols, 2):
    t = df.dropna(subset=[a, b, "体重g"]).copy()
    if len(t) < 60:
        continue
    for col in [a, b, "体重g"]:
        t[col] = np.log10(t[col].where(t[col] > 0))
    t = t.dropna(subset=[a, b, "体重g"])
    Xt = np.column_stack([np.ones(len(t)), t["体重g"]])
    res = lambda v: t[v] - Xt @ np.linalg.lstsq(Xt, t[v], rcond=None)[0]
    r_raw = t[a].corr(t[b])
    r_ctl = np.corrcoef(res(a), res(b))[0, 1]
    if abs(r_raw) > 0.4:
        mark = "  ← 消えた" if abs(r_ctl) < 0.2 else ""
        print(f"{a:<10}×{b:<10} {r_raw:+.3f}   {r_ctl:+.3f}   {len(t):>3}{mark}")

**強い相関がいくつも並び、その多くが体重を統制すると消える。**

これは統計の失敗ではなく、**世界のほうがそうできている**ということである。生き物の形と暮らしは、体の大きさという1本の軸に強く縛られている。

> だから比較生物学では、**「体サイズを統制したうえで、まだ残る関係は何か」**が本当の問いになる。
> （さらに厳密には、近縁な種どうしは似ているという問題もあり、系統関係も考慮する方法がある。第10回の回帰・第11回のモデル選択につながる話である。）

---
## 4. r だけ見るな ―― アンスコムの四重奏

有名な例。**4つのまったく違うデータ**が、どれも r = 0.816、回帰直線もほぼ同じ。でも散布図は別物。

In [ ]:
x = np.array([10,8,13,9,11,14,6,4,12,7,5], float)
x4 = np.array([8,8,8,8,8,8,8,19,8,8,8], float)
ys = {
  "I 直線的":  [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68],
  "II 曲線":   [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74],
  "III 外れ値":[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73],
  "IV 1点が支配":[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89],
}
fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for i,(name,y) in enumerate(ys.items()):
    xx = x4 if name.startswith("IV") else x
    rr = np.corrcoef(xx, y)[0,1]
    ax[i].scatter(xx, y, color="#00897b"); ax[i].set_title(f"{name}\nr={rr:.2f}")
    ax[i].set_ylim(2,13); ax[i].set_xlim(3,20)
plt.tight_layout(); plt.show()

全部 r=0.82。でも――IIは曲線、IIIは1個の外れ値、IVはたった1点が全体を支配している。
**r という1つの数だけ見て『関係あり』と報告するのは危険。必ず散布図を見る。**

> **第2回・第3回でやったことを思い出そう。**> 生の体重で散布図を描けば、ゴリラ1種が図の右端に飛び出し、> アンスコムのIVと同じ「1点が支配する」状態になる。**対数にしたのは、それを避けるためでもある。**

---
## 5. r の限界：曲がった関係は見えない

In [ ]:
xx = np.linspace(-3, 3, 100)
yy = xx**2          # きれいな放物線（強い関係）
print(f"放物線 y=x^2 の相関係数 r = {np.corrcoef(xx, yy)[0,1]:.3f}（ほぼ0！）")
plt.figure(figsize=(5,3.5))
plt.scatter(xx, yy, s=10, color="#00897b")
plt.title("明らかに関係あり。でも r ≈ 0"); plt.xlabel("x"); plt.ylabel("y")
plt.show()

r が測れるのは**直線的な**関係だけ。U字やくの字の関係は r ≈ 0 になり、「関係なし」と誤解してしまう。やはり**まず散布図**。

---
## 今日のまとめ

| 注意 | 中身 |
|---|---|
| 相関 ≠ 因果 | 一緒に動いても原因とは限らない |
| 交絡 | 隠れた第三の変数が両方を動かす（**離乳と行動圏の例＝体の大きさ**） |
| 逆因果 | 原因と結果が逆かも |
| 偶然 | たまたま相関することもある |
| r だけ見ない | アンスコム・非線形・外れ値 → 必ず散布図 |
| **尺度** | 何桁にもまたがる量は対数で見る（1点支配を避ける） |

> **相関を見たら、まず散布図を描き、次に『第三の変数』を疑う。**
> 因果を確かめるには、本当は**実験（ランダム化）**が要る ―― それは後の回で。

今日いちばん怖かったのは、**「乳離れが遅い種ほど広く動く」が、それらしく聞こえたこと**である。
ばかげた例（アイスと水難事故）は誰でも見抜ける。**見抜けないのは、もっともらしい例のほうだ。**

**課題（Moodle）**：与えられた相関事例について「因果と言えるか」「対抗仮説（交絡など）を1つ」を答える。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。